# 04 — Preprocessing and Feature Decisions

This notebook prepares the Telco Churn dataset for modeling.

Main goals:
- define the target variable
- normalize missing-like values
- identify leakage-sensitive columns
- decide which features to keep, drop, or review
- prepare feature groups for the modeling pipeline

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path
import json

In [3]:
DATA_PATH = Path("../data/raw/telco.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
display(pd.DataFrame({"column": df.columns}))

Shape: (7043, 50)

Columns:


,column
0,Customer ID
1,Gender
2,Age
3,Under 30
4,Senior Citizen
5,Married
6,Dependents
7,Number of Dependents
8,Country
9,State


## 1. Define target variable

`Churn Label` is used as the main binary target for modeling.

In [7]:
df["target_churn"] = df["Churn Label"].map({"No": 0, "Yes": 1})

df[["Churn Label", "target_churn"]].sample(7)

,Churn Label,target_churn
1839,Yes,1
238,Yes,1
1526,Yes,1
1307,Yes,1
6751,No,0
5068,No,0
2893,No,0


In [8]:
print(df["Churn Label"].value_counts(dropna=False))
print()
print(df["target_churn"].value_counts(dropna=False))

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

target_churn
0    5174
1    1869
Name: count, dtype: int64


## 2. Create a working modeling dataframe

A copy of the raw dataframe is created for preprocessing decisions.

In [9]:
model_df = df.copy()
model_df.shape

(7043, 51)

## 3. Normalize missing-like values

Earlier validation showed that some columns use literal `"None"` while others use actual null values.
These values are standardized before modeling.

In [10]:
object_cols = model_df.select_dtypes(include="object").columns.tolist()
object_cols

['Customer ID',
 'Gender',
 'Under 30',
 'Senior Citizen',
 'Married',
 'Dependents',
 'Country',
 'State',
 'City',
 'Quarter',
 'Referred a Friend',
 'Offer',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Internet Type',
 'Online Security',
 'Online Backup',
 'Device Protection Plan',
 'Premium Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Streaming Music',
 'Unlimited Data',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Customer Status',
 'Churn Label',
 'Churn Category',
 'Churn Reason']

In [11]:
model_df[object_cols] = model_df[object_cols].replace("None", np.nan)
model_df[object_cols] = model_df[object_cols].replace(r"^\s*$", np.nan, regex=True)

In [12]:
missing_summary = (
    model_df.isna()
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
missing_summary.columns = ["column", "missing_count"]

missing_summary[missing_summary["missing_count"] > 0]

,column,missing_count
0,Churn Reason,5174
1,Churn Category,5174
2,Offer,3877
3,Internet Type,1526


## 4. Identify leakage-sensitive columns

Some columns appear to contain post-outcome or highly target-proximate information.
These columns are removed from modeling features.

In [13]:
leakage_cols = [
    "Customer Status",
    "Churn Category",
    "Churn Reason",
    "Churn Score"
]

existing_leakage_cols = [col for col in leakage_cols if col in model_df.columns]
existing_leakage_cols

['Customer Status', 'Churn Category', 'Churn Reason', 'Churn Score']

## 5. Review identifier and non-modeling columns

Some columns may not be useful for baseline modeling or may require later review.

In [14]:
review_cols = [
    "Customer ID",
    "Count",
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude",
    "Quarter",
    "CLTV",
    "Total Revenue"
]

existing_review_cols = [col for col in review_cols if col in model_df.columns]
existing_review_cols

['Customer ID',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Latitude',
 'Longitude',
 'Quarter',
 'CLTV',
 'Total Revenue']

## 6. Drop non-feature columns for the baseline modeling dataset

Target-related leakage columns and selected review columns are excluded from the first modeling version.

## Column Exclusion Rationale

Some columns were removed from the baseline modeling dataset for one of the following reasons:

- leakage risk
- identifier-only role
- low analytical value
- high-cardinality geographic detail
- derived business metric requiring further review

These exclusions are intended to create a more reliable and interpretable baseline model.

In [3]:
drop_reasons = pd.DataFrame({
    "column": [
        "Customer Status",
        "Churn Category",
        "Churn Reason",
        "Churn Score",
        "Customer ID",
        "Count",
        "Country",
        "State",
        "City",
        "Zip Code",
        "Lat Long",
        "Latitude",
        "Longitude",
        "Quarter",
        "CLTV",
        "Total Revenue"
    ],
    "reason": [
        "Potential post-outcome status; leakage risk",
        "Post-churn categorical information; direct leakage",
        "Post-outcome explanatory field; direct leakage",
        "Derived churn-risk score; high leakage risk",
        "Unique identifier; not generalizable",
        "Constant / low-information field",
        "Likely constant / low-information field",
        "Low-priority location feature for baseline",
        "High-cardinality geographic field",
        "Highly granular geographic identifier",
        "Redundant combined geographic field",
        "Detailed coordinate field; unnecessary for baseline",
        "Detailed coordinate field; unnecessary for baseline",
        "Unclear baseline contribution",
        "Derived business metric requiring review",
        "Derived aggregate financial feature"
    ]
})

drop_reasons

,column,reason
0,Customer Status,Potential post-outcome status; leakage risk
1,Churn Category,Post-churn categorical information; direct lea...
2,Churn Reason,Post-outcome explanatory field; direct leakage
3,Churn Score,Derived churn-risk score; high leakage risk
4,Customer ID,Unique identifier; not generalizable
5,Count,Constant / low-information field
6,Country,Likely constant / low-information field
7,State,Low-priority location feature for baseline
8,City,High-cardinality geographic field
9,Zip Code,Highly granular geographic identifier


In [4]:
report_path = "../reports/drop_reasons.csv"
drop_reasons.to_csv(report_path, index=False)

print(f"Feature exclusion rationale saved as report: {report_path}")

Feature exclusion rationale saved as report: ../reports/drop_reasons.csv


In [15]:
drop_cols = existing_leakage_cols + existing_review_cols + ["Churn Label"]

drop_cols = [col for col in drop_cols if col in model_df.columns]

baseline_df = model_df.drop(columns=drop_cols).copy()

print("Dropped columns:")
print(drop_cols)

print("\nBaseline shape:", baseline_df.shape)

Dropped columns:
['Customer Status', 'Churn Category', 'Churn Reason', 'Churn Score', 'Customer ID', 'Country', 'State', 'City', 'Zip Code', 'Latitude', 'Longitude', 'Quarter', 'CLTV', 'Total Revenue', 'Churn Label']

Baseline shape: (7043, 36)


In [16]:
baseline_df.head()

,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Population,Referred a Friend,Number of Referrals,...,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Satisfaction Score,target_churn
0,Male,78,No,Yes,No,No,0,68701,No,0,...,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,3,1
1,Female,74,No,Yes,Yes,Yes,1,55668,Yes,1,...,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,3,1
2,Male,71,No,Yes,No,Yes,3,47534,No,0,...,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,2,1
3,Female,78,No,Yes,Yes,Yes,1,27778,Yes,1,...,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2,1
4,Female,80,No,Yes,Yes,Yes,1,26265,Yes,1,...,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,2,1


## 7. Separate features and target

The baseline dataset is split into `X` and `y` for the modeling pipeline.

In [17]:
X = baseline_df.drop(columns=["target_churn"])
y = baseline_df["target_churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 35)
y shape: (7043,)


In [18]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include="object").columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 13
Categorical features: 22


In [19]:
print("Numeric columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

Numeric columns:
['Age', 'Number of Dependents', 'Population', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Charges', 'Total Refunds', 'Total Extra Data Charges', 'Total Long Distance Charges', 'Satisfaction Score']

Categorical columns:
['Gender', 'Under 30', 'Senior Citizen', 'Married', 'Dependents', 'Referred a Friend', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method']


## 8. Final preprocessing notes

At this stage:
- the target is defined
- missing-like values are standardized
- leakage-sensitive columns are excluded
- feature groups are ready for a scikit-learn preprocessing pipeline

The next notebook will focus on train/test split, preprocessing pipeline construction, and model evaluation.

In [20]:
feature_config = {
    "target_column": "target_churn",
    "dropped_columns": drop_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}

output_path = Path("../reports/feature_config.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=4)

print(f"Saved to: {output_path}")

Saved to: ..\reports\feature_config.json


## Conclusion

The dataset is now prepared for baseline modeling.
Target definition, missing normalization, and feature selection decisions have been completed.
The next step is to build a reproducible modeling pipeline and compare classification models.